# Provincial Household Debt Prediction

This notebook trains a supervised machine-learning model to estimate a province's **average household debt** from its average monthly income, expenses, expense-to-income ratio, and region.

The modeling table contains one row per province (77 rows). The raw CSV has 3,388 rows, but those rows repeat provinces across indicators and socioeconomic categories, so the raw rows must not be randomly split as if they were independent observations.

## Modeling scope

- **Model type:** supervised regression
- **Target:** `debt_average_baht`
- **Features:** monthly income, monthly expenses, expense-to-income percentage, and region
- **Validation:** shuffled 5-fold cross-validation with `random_state=42`
- **Model:** Ridge regression with `alpha=10.0`
- **Metrics:** MAE, RMSE, and R²

This is a province-level estimation exercise for B.E. 2566 (C.E. 2023). It does not predict individual household debt, establish causality, or forecast future years.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

data_path_candidates = [
    Path('data/raw/SFD_SPB0801.csv'),
    Path('../data/raw/SFD_SPB0801.csv'),
]
data_path = next((path for path in data_path_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find data/raw/SFD_SPB0801.csv. Run this notebook from the repository root.')
-
raw_df = pd.read_csv(data_path)
metric_column = 'mthincome_mthexp_totaldebt_pctexptoincome'
df = raw_df.rename(columns={metric_column: 'indicator'}).copy()
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df['year_ce'] = df['year'] - 543

TOTAL_LABEL = 'รวมทั้งสิ้น'
print(f'Loaded: {data_path}')
print(f'Raw shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Year: {sorted(df["year"].unique().tolist())}')
print(f'Provinces: {df["province"].nunique()}')

In [ ]:
# Keep the overall total for each province and pivot indicators into feature columns.
total_scope = df.loc[
    (df['soc_eco_class1'] == TOTAL_LABEL) & (df['soc_eco_class2'] == TOTAL_LABEL)
].copy()

wide = total_scope.pivot(
    index=['year', 'year_ce', 'province'],
    columns='indicator',
    values='value',
).reset_index()
wide.columns.name = None
wide = wide.rename(columns={
    'รายได้ทั้งสิ้นต่อเดือน': 'income_monthly_baht',
    'ค่าใช้จ่ายทั้งสิ้นต่อเดือน': 'expense_monthly_baht',
    'หนี้สินเฉลี่ยต่อครัวเรือนทั้งสิ้น': 'debt_average_baht',
    'ร้อยละของค่าใช้จ่ายต่อรายได้': 'expense_income_pct',
})

REGION_PROVINCES = {
    'กรุงเทพมหานครและ 3 จังหวัด': ['กรุงเทพมหานคร', 'นนทบุรี', 'ปทุมธานี', 'สมุทรปราการ'],
    'ภาคกลาง': [
        'กาญจนบุรี', 'จันทบุรี', 'ฉะเชิงเทรา', 'ชลบุรี', 'ชัยนาท', 'ตราด', 'นครนายก',
        'นครปฐม', 'ประจวบคีรีขันธ์', 'ปราจีนบุรี', 'พระนครศรีอยุธยา', 'เพชรบุรี', 'ระยอง',
        'ราชบุรี', 'ลพบุรี', 'สมุทรสงคราม', 'สมุทรสาคร', 'สระแก้ว', 'สระบุรี', 'สิงห์บุรี',
        'สุพรรณบุรี', 'อ่างทอง',
    ],
    'ภาคเหนือ': [
        'กำแพงเพชร', 'เชียงราย', 'เชียงใหม่', 'ตาก', 'นครสวรรค์', 'น่าน', 'พะเยา',
        'พิจิตร', 'พิษณุโลก', 'เพชรบูรณ์', 'แพร่', 'แม่ฮ่องสอน', 'ลำปาง', 'ลำพูน',
        'สุโขทัย', 'อุตรดิตถ์', 'อุทัยธานี',
    ],
    'ภาคตะวันออกเฉียงเหนือ': [
        'กาฬสินธุ์', 'ขอนแก่น', 'ชัยภูมิ', 'นครพนม', 'นครราชสีมา', 'บึงกาฬ', 'บุรีรัมย์',
        'มหาสารคาม', 'มุกดาหาร', 'ยโสธร', 'ร้อยเอ็ด', 'เลย', 'ศรีสะเกษ', 'สกลนคร',
        'สุรินทร์', 'หนองคาย', 'หนองบัวลำภู', 'อุดรธานี', 'อุบลราชธานี', 'อำนาจเจริญ',
    ],
    'ภาคใต้': [
        'กระบี่', 'ชุมพร', 'ตรัง', 'นครศรีธรรมราช', 'นราธิวาส', 'ปัตตานี', 'พังงา',
        'พัทลุง', 'ภูเก็ต', 'ยะลา', 'ระนอง', 'สงขลา', 'สตูล', 'สุราษฎร์ธานี',
    ],
}

province_to_region = {
    province: region
    for region, provinces in REGION_PROVINCES.items()
    for province in provinces
}
wide['region'] = wide['province'].map(province_to_region)
wide['region'] = pd.Categorical(wide['region'], categories=list(REGION_PROVINCES), ordered=True)

required_columns = [
    'income_monthly_baht',
    'expense_monthly_baht',
    'debt_average_baht',
    'expense_income_pct',
    'region',
]
assert len(wide) == 77
assert wide[required_columns].notna().all().all()
assert wide['region'].nunique() == 5

print(f'Modeling rows: {len(wide)} provinces')
print('Rows by region:')
print(wide.groupby('region', observed=True)['province'].nunique().to_string())
wide.head()

## K-fold cross-validation

Cross-validation happens after reshaping to one row per province. With 5-fold cross-validation, each province is used in a validation fold once and in the training folds four times. `province` itself is not used as a feature because a unique province label would let the model memorize the training rows instead of learning a general relationship. Region is retained as a broader categorical feature.

In [ ]:
features = [
    'income_monthly_baht',
    'expense_monthly_baht',
    'expense_income_pct',
    'region',
]
target = 'debt_average_baht'

model_df = wide[features + [target]].dropna().copy()
X = model_df[features]
y = model_df[target]

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print(f'Total modeling rows: {len(X)} provinces')
print(f'Cross-validation folds: {cv.n_splits}')
print('Each validation fold contains approximately 15–16 provinces.')

In [ ]:
numeric_features = [
    'income_monthly_baht',
    'expense_monthly_baht',
    'expense_income_pct',
]
categorical_features = ['region']

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=10.0)),
])

# Out-of-fold predictions ensure every prediction is made for a row
# that was excluded from the fold used to train that prediction.
predictions = cross_val_predict(model, X, y, cv=cv)

# Fit once on all provinces for coefficient inspection later.
# The cross-validation predictions above remain the evaluation predictions.
model.fit(X, y)

print('Model configuration:')
print('- Estimator: Ridge regression')
print('- Regularization: alpha=10.0')
print('- Numeric preprocessing: median imputation + standardization')
print('- Region preprocessing: one-hot encoding')

In [ ]:
# Compare the model with a simple median-prediction baseline using the same folds.
baseline = DummyRegressor(strategy='median')
baseline_predictions = cross_val_predict(baseline, X, y, cv=cv)

def regression_metrics(actual, predicted):
    return {
        'MAE_baht': mean_absolute_error(actual, predicted),
        'RMSE_baht': mean_squared_error(actual, predicted) ** 0.5,
        'R2': r2_score(actual, predicted),
    }

results = pd.DataFrame({
    'Ridge model': regression_metrics(y, predictions),
    'Median baseline': regression_metrics(y, baseline_predictions),
}).T

print('Out-of-fold cross-validation evaluation:')
results.round(3)

In [ ]:
# Inspect out-of-fold predictions and visualize actual versus predicted debt.
prediction_table = X.copy()
prediction_table['actual_debt_baht'] = y
prediction_table['predicted_debt_baht'] = predictions
prediction_table['absolute_error_baht'] = (
    prediction_table['actual_debt_baht'] - prediction_table['predicted_debt_baht']
).abs()
prediction_table = (
    wide[['province']]
    .join(prediction_table, how='right')
    .sort_values('absolute_error_baht', ascending=False)
)
print('Out-of-fold predictions, sorted by absolute error:')
display(prediction_table)

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    x=y,
    y=predictions,
    s=80,
    color='#4C78A8',
    ax=ax,
)
min_value = min(y.min(), predictions.min())
max_value = max(y.max(), predictions.max())
ax.plot([min_value, max_value], [min_value, max_value], '--', color='#E45756')
ax.set_title('Actual versus predicted provincial average debt')
ax.set_xlabel('Actual debt (baht)')
ax.set_ylabel('Predicted debt (baht)')
plt.tight_layout()
plt.show()

In [ ]:
# Ridge coefficients provide an interpretable view of the fitted model.
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficient_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient_baht': model.named_steps['regressor'].coef_,
})
coefficient_table['absolute_coefficient'] = coefficient_table['coefficient_baht'].abs()
coefficient_table.sort_values('absolute_coefficient', ascending=False).drop(columns='absolute_coefficient')

## Interpretation and limitations

A lower MAE or RMSE than the median baseline means Ridge performed better across the out-of-fold predictions. R² can be negative, which means the model did worse than a simple mean-based prediction.

The result should be interpreted cautiously because there are only 77 province-level observations from one year. Each validation fold contains only about 15–16 provinces, so metrics can change with another random seed or fold configuration. `expense_income_pct` is also derived from income and expenses, so a stricter version of the experiment could rerun the model without that feature.

Most importantly, this model estimates **average provincial debt**, not the debt of an individual household. It describes association in the published 2023 data and does not demonstrate that income or expenses cause debt.